In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mnn_torch import paths

# The committed 20-seed grids are batch-produced (see the last cell); the cells below
# always REPLAY them, so every figure renders instantly. RES/SR point at the two grid
# trees under data/results/ (this replaces the source generators' `../results` path).
RES = str(paths.results_dir() / "temporal")
SR = str(paths.results_dir() / "storerecall")

# Palette lifted verbatim from gen_results.py / gen_storerecall.py.
TEAL = "#3aa07a"; RED = "#c0392b"; BLUE = "#2f4b8f"; GREY = "#9aa6b2"; AMBER = "#e0a93b"
INDIGO = "#5b6fb0"; INK = "#2b2b2b"


def boot_ci(v, n=10000, seed=0):
    v = np.asarray(v, float); rng = np.random.default_rng(seed)
    b = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(n)]
    return np.percentile(b, 2.5), np.percentile(b, 97.5)


print("data/results:", paths.results_dir())
print("temporal:", RES)
print("storerecall:", SR)

In [ ]:
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch, Rectangle
from matplotlib.lines import Line2D

MEMBG = "#eaf0fb"; LIFBG = "#eaf5ef"


def neuron_col(ax, x, ys, r, fc, ec):
    for y in ys:
        ax.add_patch(Circle((x, y), r, fc=fc, ec=ec, lw=1.2, zorder=3))


def connect(ax, x0, ys0, x1, ys1, color, lw=0.5, alpha=0.5):
    for y0 in ys0:
        for y1 in ys1:
            ax.plot([x0, x1], [y0, y1], color=color, lw=lw, alpha=alpha, zorder=1)


def crossbar(ax, x, y, w, h, n=4, color=BLUE):
    '''A small RRAM crossbar glyph: horizontal wordlines, vertical bitlines, memristor
    dots at crosspoints.'''
    xs = np.linspace(x + 0.12 * w, x + 0.88 * w, n)
    ys = np.linspace(y + 0.15 * h, y + 0.85 * h, n)
    for yy in ys:
        ax.plot([x + 0.06 * w, x + 0.94 * w], [yy, yy], color=GREY, lw=0.9, zorder=2)
    for xx in xs:
        ax.plot([xx, xx], [y + 0.06 * h, y + 0.94 * h], color=GREY, lw=0.9, zorder=2)
    for xx in xs:
        for yy in ys:
            ax.add_patch(Circle((xx, yy), 0.012, fc=color, ec="none", zorder=3))


def membrane_glyph(ax, x, y, w, h, color=TEAL):
    '''Leaky-integrator glyph: a rise then exponential decay inside a rounded panel.'''
    t = np.linspace(0, 1, 120)
    rise = 1 - np.exp(-t / 0.06)
    yv = rise * np.exp(-t / 0.5); yv = yv / yv.max()
    ax.plot(x + 0.1 * w + t * 0.8 * w, y + 0.2 * h + yv * 0.6 * h,
            color=color, lw=2.0, zorder=3)


fig = plt.figure(figsize=(10, 7.2))
axA = fig.add_axes([0.02, 0.53, 0.96, 0.42]); axA.axis("off")
axB = fig.add_axes([0.02, 0.03, 0.96, 0.42]); axB.axis("off")
for ax in (axA, axB):
    ax.set_ylim(0.15, 4)
axA.set_xlim(0.1, 8.4)   # panel (a) content ends ~8.0 (caption); tight -> no whitespace
axB.set_xlim(0.1, 9.4)   # panel (b) runs further (delayed-readout marker + label)

# ===================== (a) TemporalMSNN (dense) =====================
axA.text(0.0, 3.9, "(a) TemporalMSNN --- dense streamed network",
         fontsize=13.5, color=INK, fontweight="bold")
yc = 2.0
# streamed input: an unrolled-time strip of input vectors
axA.text(1.05, 3.35, "streamed input  $x[1],\\,x[2],\\dots,x[T]$", fontsize=12,
         color=INK, ha="center")
for k, off in enumerate([-0.18, 0.0, 0.18]):
    axA.add_patch(FancyBboxPatch((0.55 + k * 0.16, 1.2 + k * 0.0), 0.28, 1.5,
                  boxstyle="round,pad=0.01", fc="#f4f6f9", ec=GREY, lw=1.1, zorder=2))
in_ys = np.linspace(1.4, 2.6, 4)
neuron_col(axA, 1.05, in_ys, 0.10, "white", GREY)
# memristive crossbar (FC synapse)
axA.add_patch(FancyBboxPatch((2.1, 1.05), 1.7, 1.9, boxstyle="round,pad=0.02",
              fc=MEMBG, ec=BLUE, lw=1.4, zorder=1))
crossbar(axA, 2.1, 1.05, 1.7, 1.9, n=4, color=BLUE)
axA.text(2.95, 0.82, "memristive crossbar\n(Poole--Frenkel synapse)", fontsize=11.5,
         color=BLUE, ha="center", va="top")
connect(axA, 1.15, in_ys, 2.2, np.linspace(1.4, 2.6, 4), GREY)
# DeviceLeaky LIF hidden layer
hid_ys = np.linspace(1.35, 2.65, 5)
axA.add_patch(FancyBboxPatch((4.25, 1.0), 1.9, 2.0, boxstyle="round,pad=0.02",
              fc=LIFBG, ec=TEAL, lw=1.4, zorder=1))
neuron_col(axA, 4.75, hid_ys, 0.11, "white", TEAL)
membrane_glyph(axA, 5.15, 1.35, 0.9, 1.3, TEAL)
axA.text(5.2, 3.05, "DeviceLeaky LIF,  $\\beta=e^{-\\Delta t/\\tau_{\\rm leak}}$",
         fontsize=11.5, color=TEAL, ha="center", va="bottom")
connect(axA, 3.75, np.linspace(1.4, 2.6, 4), 4.65, hid_ys, GREY, alpha=0.35)
# readout
out_ys = np.linspace(1.7, 2.3, 2)
neuron_col(axA, 7.1, out_ys, 0.12, "white", INK)
connect(axA, 4.85, hid_ys, 7.0, out_ys, GREY, alpha=0.35)
axA.text(7.1, 1.35, "readout", fontsize=11.5, color=INK, ha="center")
# recurrent membrane-across-time loop (compact, under the LIF block)
axA.add_patch(FancyArrowPatch((5.6, 1.0), (5.6, 0.6), arrowstyle="-", color=TEAL, lw=1.4, zorder=4))
axA.add_patch(FancyArrowPatch((5.6, 0.6), (4.6, 0.6), arrowstyle="-", color=TEAL, lw=1.4, zorder=4))
axA.add_patch(FancyArrowPatch((4.6, 0.6), (4.6, 1.0), arrowstyle="-|>", color=TEAL,
              lw=1.4, mutation_scale=12, zorder=4))
# caption directly BELOW the green LIF box (box centred at x=5.2), under the loop
axA.text(5.1, 0.28, "membrane state carried across $t$", fontsize=11, color=TEAL,
         ha="center", va="center", style="italic")
# flow arrows between stages
for x0, x1 in [(1.25, 2.05), (3.85, 4.2), (6.2, 6.95)]:
    axA.add_patch(FancyArrowPatch((x0, yc), (x1, yc), arrowstyle="-|>", color=INK,
                  lw=1.3, mutation_scale=13, zorder=5))

# ===================== (b) TemporalMCSNN (conv) =====================
axB.text(0.0, 3.9, "(b) TemporalMCSNN --- convolutional streamed network",
         fontsize=13.5, color=INK, fontweight="bold")
# streamed clip: stacked frames
axB.text(1.0, 3.35, "streamed clip  (frames)", fontsize=12, color=INK, ha="center")
for k, off in enumerate([0.0, 0.13, 0.26]):
    axB.add_patch(Rectangle((0.45 + off, 1.35 + off), 0.85, 0.85, fc="#f4f6f9",
                  ec=GREY, lw=1.1, zorder=2 + k))
axB.text(1.0, 1.1, "$C\\times H\\times W$", fontsize=11, color=INK, ha="center", va="top")
# memristive conv stack (two feature-map stacks + a crossbar hint)
def fmap_stack(ax, x, y, n, w, h, color):
    for k in range(n):
        ax.add_patch(Rectangle((x + k * 0.10, y + k * 0.10), w, h, fc=MEMBG, ec=color,
                     lw=1.1, zorder=2 + k))
fmap_stack(axB, 2.2, 1.4, 3, 0.7, 0.7, BLUE)
fmap_stack(axB, 3.3, 1.5, 3, 0.55, 0.55, BLUE)
axB.text(3.0, 0.95, "memristive conv $\\times 2$\n(Poole--Frenkel)", fontsize=11.5,
         color=BLUE, ha="center", va="top")
# DeviceLeaky LIF
hid_ys = np.linspace(1.4, 2.6, 5)
axB.add_patch(FancyBboxPatch((4.9, 1.0), 1.9, 2.0, boxstyle="round,pad=0.02",
              fc=LIFBG, ec=TEAL, lw=1.4, zorder=1))
neuron_col(axB, 5.4, hid_ys, 0.11, "white", TEAL)
membrane_glyph(axB, 5.8, 1.35, 0.9, 1.3, TEAL)
axB.text(5.85, 3.05, "DeviceLeaky LIF,  $\\beta=e^{-\\Delta t/\\tau_{\\rm leak}}$",
         fontsize=11.5, color=TEAL, ha="center", va="bottom")
# readout
out_ys = np.linspace(1.7, 2.3, 2)
neuron_col(axB, 7.8, out_ys, 0.12, "white", INK)
connect(axB, 5.5, hid_ys, 7.7, out_ys, GREY, alpha=0.35)
axB.text(7.8, 1.35, "readout", fontsize=11.5, color=INK, ha="center")
# delayed-readout marker
axB.add_patch(FancyArrowPatch((8.05, 2.0), (9.0, 2.0), arrowstyle="-|>", color=RED,
              lw=1.3, mutation_scale=12, ls=(0, (4, 2))))
axB.text(9.02, 1.55, "decision read\nafter delay $D$", fontsize=10.5, color=RED, ha="right")
# recurrence across frames (compact, under the LIF block)
axB.add_patch(FancyArrowPatch((6.25, 1.0), (6.25, 0.6), arrowstyle="-", color=TEAL, lw=1.4, zorder=4))
axB.add_patch(FancyArrowPatch((6.25, 0.6), (5.25, 0.6), arrowstyle="-", color=TEAL, lw=1.4, zorder=4))
axB.add_patch(FancyArrowPatch((5.25, 0.6), (5.25, 1.0), arrowstyle="-|>", color=TEAL,
              lw=1.4, mutation_scale=12, zorder=4))
# caption directly BELOW the green LIF box (box centred at x=5.85), under the loop
axB.text(5.75, 0.28, "membrane state carried across frames", fontsize=11, color=TEAL,
         ha="center", va="center", style="italic")
for x0, x1 in [(1.45, 2.05), (4.0, 4.85), (6.85, 7.65)]:
    axB.add_patch(FancyArrowPatch((x0, 2.0), (x1, 2.0), arrowstyle="-|>", color=INK,
                  lw=1.3, mutation_scale=13, zorder=5))

plt.show()

In [ ]:
caps = [("seq-MNIST\n(dense)", "cap-dense", 10.0),
        ("N-MNIST\n(conv)", "cap-conv", 10.0),
        ("SHD\n(dense)", "cap-shd", 5.0),
        ("DVS Gesture\n(conv)", "cap-dvsg", 9.1)]
names, accs, los, his, chances = [], [], [], [], []
for nm, d, ch in caps:
    z = np.load(f"{RES}/{d}/temporal_sweep.npy", allow_pickle=True).item()
    arr = list(z["acc"].values())[0][:, 0]
    lo, hi = boot_ci(arr, seed=1)
    names.append(nm); accs.append(arr.mean()); los.append(lo); his.append(hi); chances.append(ch)
fig, ax = plt.subplots(figsize=(6.0, 4.0))
x = np.arange(len(names))
err = np.array([[a - l for a, l in zip(accs, los)], [h - a for a, h in zip(accs, his)]])
ax.bar(x, accs, color=[BLUE, TEAL, BLUE, TEAL], edgecolor=INK, linewidth=0.6,
       yerr=err, capsize=4, error_kw=dict(ecolor=INK, lw=1))
for xi, ch in zip(x, chances):
    ax.plot([xi - 0.4, xi + 0.4], [ch, ch], ls=":", color=GREY, lw=1.2)
for xi, a in zip(x, accs):
    ax.text(xi, a + 2, f"{a:.1f}", ha="center", fontsize=8, color=INK)
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=8.5)
ax.set_ylabel("test accuracy (%)", fontsize=9); ax.set_ylim(0, 100)
ax.set_title("Benchmark capability (20 seeds; dotted = chance)", fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
def _retention_panel(ax, npy, title, key_suffix):
    '''Plot accuracy-vs-gap per tau for a retention grid (dense uses bare-tau keys,
    conv uses tau|0 keys).'''
    z = np.load(npy, allow_pickle=True).item()
    gaps = np.array(z["gaps"], float); chance = z["chance"]; crit = z["criterion"]
    real = sorted([float(k.split("|")[0]) for k in z["acc"] if float(k.split("|")[0]) > 0.05])
    cmap = plt.cm.viridis
    for i, t in enumerate(real):
        k = f"{t:g}{key_suffix}"
        g = z["acc"][k]; m = g.mean(0)
        lo = np.array([boot_ci(g[:, j], seed=j)[0] for j in range(len(gaps))])
        hi = np.array([boot_ci(g[:, j], seed=j)[1] for j in range(len(gaps))])
        c = cmap(i / (len(real) - 1))
        lab = f"$\\tau={t:g}$"
        ax.plot(gaps, m, "-o", ms=3, color=c, label=lab)
        ax.fill_between(gaps, lo, hi, color=c, alpha=0.12)
    nk = f"0.02{key_suffix}"
    if nk in z["acc"]:
        ax.plot(gaps, z["acc"][nk].mean(0), "--", color=RED, lw=1.4, label="no memory")
    ax.axhline(chance, ls=":", color=GREY)
    ax.axhline(crit, ls="--", color="0.4", lw=0.8)
    ax.set_xlabel("delay $D$ (blank steps)", fontsize=12)
    ax.set_ylabel("test accuracy (%)", fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.tick_params(labelsize=10.5)


fig, (axd, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))
_retention_panel(axd, f"{RES}/ret-dense/temporal_sweep.npy",
                 "(a) seq-MNIST (dense, 20 seeds)", "")
_retention_panel(ax2, f"{RES}/ret-conv-full/temporal_sweep.npy",
                 "(b) N-MNIST (conv, 20 seeds)", "|0")
# One shared legend below both panels: the per-tau fan fills the upper-right of each
# axes, so an in-axes legend overlaps the curves. Panel (b) sweeps the full tau range,
# so its handles are the complete set.
handles, labels = ax2.get_legend_handles_labels()
fig.legend(handles, labels, fontsize=9.5, ncol=5, frameon=False,
           loc="lower center", bbox_to_anchor=(0.5, -0.02),
           columnspacing=1.6, handlelength=1.8)
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

In [ ]:
# representative timescales to draw. Lead with the MEASURED band (held-bias median 1.3 and
# the upper measured edge 3.0), then a longer retention (20) shown lighter as the
# fabrication-target/plateau reference -- so the figure foregrounds the measured device
# rather than the extreme tau. (tau/D are abstract time-step units, not seconds.)
REPS = [1.3, 8.0, 20.0]
REP_COLS = {1.3: TEAL, 8.0: INDIGO, 20.0: AMBER}       # measured bold (cool), target amber (warm)
REP_ALPHA = {1.3: 1.0, 8.0: 1.0, 20.0: 0.9}


def _load(panel):
    '''Return (acc-dict keyed (arm,tau|None), delays, chance, criterion) for a panel dir,
    or None if absent. Handles both arm|tau (ideal) and arm|tau|fp (fault) key forms.'''
    import os
    path = os.path.join(SR, panel, "storerecall.npy")
    if not os.path.exists(path):
        return None
    z = np.load(path, allow_pickle=True).item()
    acc = {}
    for k, g in z["acc"].items():
        parts = k.split("|")
        arm, tau = parts[0], parts[1]
        acc[(arm, None if tau == "" else float(tau))] = g
    return acc, np.asarray(z["delays"], float), z["chance"], z["criterion"]


def _panel(ax, data, title):
    acc, delays, chance, crit = data
    # fast control (no-memory) -- red dashed, matching Fig 4 no-memory convention
    if ("lif-fast", None) in acc:
        m = acc[("lif-fast", None)].mean(0)
        ax.plot(delays, m, "--", marker="s", ms=3, color=RED, lw=1.4,
                label="fast LIF (no memory)")
    for t in REPS:
        col = REP_COLS[t]; al = REP_ALPHA[t]
        tag = " (meas.)" if t <= 10.0 else " (target)"   # field-free band extends to ~10 s
        if ("device", t) in acc:
            g = acc[("device", t)]; m = g.mean(0)
            lo = np.array([boot_ci(g[:, j], seed=j)[0] for j in range(len(delays))])
            hi = np.array([boot_ci(g[:, j], seed=j)[1] for j in range(len(delays))])
            ax.plot(delays, m, "-", marker="o", ms=3, color=col, alpha=al,
                    label=f"device $\\tau_{{\\mathrm{{leak}}}}={t:g}${tag}")
            ax.fill_between(delays, lo, hi, color=col, alpha=0.10 * al)
        if ("alif", t) in acc:
            m = acc[("alif", t)].mean(0)
            ax.plot(delays, m, "--", marker="^", ms=3, color=col, alpha=0.85 * al,
                    label=f"ALIF $\\tau_a={t:g}$")
    ax.axhline(chance, ls=":", color=GREY)          # chance (matches Fig 4)
    ax.axhline(crit, ls="--", color="0.4", lw=0.8)   # criterion (matches Fig 4)
    ax.grid(True, which="major", ls="-", lw=0.5, color="0.88", zorder=0)
    ax.set_axisbelow(True)                             # grid behind the data
    ax.set_ylabel("recall accuracy (%)", fontsize=12)
    ax.set_title(title, fontsize=13, loc="left")
    ax.set_ylim(45, 108)                               # a little headroom so curves clear the top
    ax.tick_params(labelsize=10.5)
    # per-panel legend removed: a single shared legend is added in main() to avoid the
    # overlap with the data (device tau=3 collapses into the lower-left in panel b).


# three-rung synapse ablation ladder: ideal -> healthy device-PF -> device-PF + faults
ideal = _load("ideal-only")
healthy = _load("pf-healthy")
fault = _load("fault-only")
if ideal is None:
    raise SystemExit(f"ideal store-recall grid not found under {SR}/ideal-only/")
rungs = [(ideal, "(a) ideal synapse")]
if healthy is not None:
    rungs.append((healthy, "(b) device synapse, healthy"))
if fault is not None:
    rungs.append((fault, "(c) device synapse, faulty"))
n = len(rungs)
# stack the ladder VERTICALLY (one rung per row), tall panels so each is easy to read;
# a single shared legend sits directly beneath the stack. constrained_layout packs the
# panels + legend without a reserved dead band.
fig, axes = plt.subplots(n, 1, figsize=(6.6, 3.5 * n), sharex=True,
                         constrained_layout=True)
if n == 1:
    axes = [axes]
for ax, (data, title) in zip(axes, rungs):
    _panel(ax, data, title)
    ax.set_xlabel("")                              # shared x-axis: label only the bottom
axes[-1].set_xlabel("store--recall delay $D$ (blank steps)", fontsize=12)
# single shared legend directly under the stack (entries identical across panels;
# placing it outside the axes avoids overlap with the collapsed device tau=3 curve).
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, fontsize=10.5, frameon=False, ncol=3,
           loc="outside lower center")
plt.show()
print(f"store--recall: {n}-panel vertical ladder")

In [ ]:
# Uncomment to launch the homeostasis gate sweep as a subprocess (in-kernel-safe --quick):
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "mnn_torch.training", "--gate", "--quick"])
print("see the markdown above for the full-scale batch commands")